### Set up

In [29]:
# imports

from explore_import import  *
import tpp_preprocess as tpp
import hpp_checker as hpp
import data_preprocess as dt

import pyteomics.auxiliary as aux
from pathlib import Path
import os, re, subprocess
warnings.simplefilter(action='ignore', category=FutureWarning)

In [33]:
# base directories

root="/project/def-marie87/vvshazia/pride_reanalysis/CompOmics/oui-discovery"
data_dir=f"{root}/oui-discovery-vv-data/raw/tpp_pride_reanalysis/tpp_pride_reanalysis_arch230525"
processed_dir=f"{root}/oui-discovery-vv-data/processed/tpp_pride_reanalysis/tpp_pride_reanalysis_arch230525"
r_path="Rscript"
gwalk_work_dir=root
gwalk_script="Run_group_walk_tppfragpipe.R"

In [35]:
# replicate insides of data directory to preprocessed directory

#get insides of data directory
data_paths=dt.list_files(data_dir)

#replicate
for parent in data_paths.keys():
    new_parent=parent.replace(data_dir, processed_dir)
    os.makedirs(new_parent, exist_ok=True)

tpp_pride_reanalysis_arch230525/
    PXD014258/
        PXD014258-openprot/
            ESC-HF-Sample-MCF_openprot_PeptideProphet.pep.xml.index
            ESC-HF-Sample-MCF5.RAW.pep.xml
            ESC-HF-Sample-BT474_1_RAW.pep.xml
            ESC-HF-SampleHela_openprot_ProteinProphet.pep.prot.xml
            ESC-HF-SampleHela5.RAW.pep.xml
            ESC-HF-SampleHela_openprot_PeptideProphet.pep.xml.index
            ESC-HF-Sample-BT474_3_RAW.pep.xml
            ESC-HF-Sample-BT474_5_RAW.pep.xml
            ESC-HF-SampleHela3.RAW.pep.xml
            ESC-HF-Sample-MCF1.RAW.pep.xml
            ESC-HF-Sample-BT474_openprot_PeptideProphet.pep.xml
            ESC-HF-SampleHela1.RAW.pep.xml
            ESC-HF-Sample-MCF3.RAW.pep.xml
            ESC-HF-Sample-BT474_openprot_ProteinProphet.pep.prot.xml
            ESC-HF-SampleHela2.RAW.pep.xml
            ESC-HF-Sample-MCF_openprot_ProteinProphet.pep.prot.xml
            ESC-HF-Sample-MCF_openprot_PeptideProphet.pep.xml
            ESC-HF-S

### Load and process PeptideProphet files

In [13]:
def classify_leadprot(x):
    x=x.replace("decoy_","")
    if 'CONTAMINANT' in x.upper():
        return 'Contam'
    elif x.startswith('II_') or x.startswith('IP_'):
        return 'NonCanon'
        # Ensembl is canonical
    else:
        return 'Canon'

def is_peptide_canonical(x):
    '''x is the list of protein classes'''
    if np.array([_=='Contam' for _ in x]).any():
        return 'Contam'
    if np.array([_=='Canon' for _ in x]).any():
        return 'Canonical'
    return 'NonCanonical'

def classifiy_mods(row):
    if len(row.modifications)==0:
        return 'Unmodified'
    else:
        return 'Expected'

def custom_subgroup_filter(data_, key):
    filtered_subgroups = []
    for (c,m),df in data_.groupby(['isCanonical','isModified']).__iter__():
        tmp = aux.target_decoy.qvalues(df, key=key, reverse=True, is_decoy=df.database=='D',
                                      formula=1, full_output=True, q_label='custom_q')
        filtered_subgroups.append(tmp)

    return pd.concat(filtered_subgroups, ignore_index=True)

In [46]:
def col_isna(df,col):
    return 100-round((df[col].isna().value_counts()[False]/len(data))*100,4)

In [57]:
qval_score='peptideprophet_probability'

for parent, files in data_paths.items():
    new_parent=parent.replace(data_dir,processed_dir)
    dataset, database = parent.split("/")[-1].split("-")
    for file in files:
        if file.endswith("_PeptideProphet.pep.xml"):
            # read PeptideProphet output
            pepxml_path=os.path.join(parent, file)
            data=pepxml.DataFrame(pepxml_path)
            print(f"{dataset}|{database}|{file} N of discoveries: {len(data)}")
            
            #pin dataset and search_database and spectrum_file
            data["dataset"] = dataset
            data["search_database"] = database
            data["spectrum_file"] = file.replace("_PeptideProphet.pep.xml","").replace(f"_{database}","")
            
            #pin T/D database
            data["database"]=data["protein"].apply(tpp.get_database_tpp)
            td_counts = round((data["database"].value_counts()/len(data))*100,4)
            print(f"{dataset}|{database}|{file} targets and decoys and NA: {td_counts['T']} and {td_counts['D']} and {col_isna(data,'database')}")

            #calculate global q-value on peptideprophet_probability
            print(f"{dataset}|{database}|{file} {qval_score} range and NA: {min(data[qval_score])} - {max(data[qval_score])} and {col_isna(data,qval_score)}")
            data = aux.target_decoy.qvalues(data,
                                            key=qval_score,
                                            reverse=True,
                                            is_decoy=(data.database == 'D'),
                                            q_label='global_q',
                                            formula=1,
                                            full_output=True)
            print(f"{dataset}|{database}|{file} 'global_q' range and NA: {min(data['global_q'])} - {max(data['global_q'])} and {col_isna(data,'global_q')}")

            #pin helpfull labels
            data["peptide_class"]=data["protein"].apply(tpp.classify_peptide_tpp)
            
            #pin subgroups
            data['protein_classes'] = data.protein.apply(lambda x: np.unique([classify_leadprot(_) for _ in x]))
            data['isCanonical'] = data.protein_classes.apply(is_peptide_canonical)
            data['isModified']  = data.apply(classifiy_mods,axis=1)
        
            # Adds a columns telling you if a PSM passes the custom filters or not (custom_filter_pass)
            data = custom_subgroup_filter(data, qval_score)
            print(f"{dataset}|{database}|{file} 'custom_q' range and NA: {min(data['custom_q'])} - {max(data['custom_q'])} and {col_isna(data,'custom_q')}")
            #add hybrid/group-wise
            data["glob_cust_hybrid"]=data.apply(lambda x: x.custom_q if x.isCanonical=="NonCanonical" else x.global_q, axis=1)
            print(f"{dataset}|{database}|{file} 'glob_cust_hybrid' range and NA: {min(data['glob_cust_hybrid'])} - {max(data['glob_cust_hybrid'])} and {col_isna(data,'glob_cust_hybrid')}")
        
            # prepare for Group-walk
            data['isTarget'] = data.database.apply(lambda x: x=='T')
            data['FDRGroup'] = data.isCanonical + '_' + data.isModified
            fdrgroup_counts = round((data["FDRGroup"].value_counts()/len(data))*100,4)
            counts_str = ", ".join(
                f"{key} {fdrgroup_counts[key]}%"
                for key in list(fdrgroup_counts.keys())
            )
            na_count = col_isna(data, 'FDRGroup')
            print(f"{dataset}|{database}|{file} {counts_str}, NA {na_count}")
            #save for Group-walk
            basename = os.path.basename(pepxml_path).split('.')[0]
            gwalk_input = os.path.join(new_parent, f"{basename}.csv")
            data.to_csv(gwalk_input)
            
            #run Group-walk
            dataset_dir = new_parent
            working_dir = gwalk_work_dir
            file_name = os.path.basename(gwalk_input)
            print(f"Run Group-walk on {gwalk_input}")
            command = (
                f"module load r && {r_path} {gwalk_script} {dataset_dir} {working_dir} {file_name}"
            )
            
            _ = subprocess.run(command, shell=True, check=True)
            
            qwalk_output = os.path.join(new_parent,"groupwalk_output_"+file_name)
            
            #add hybrid/group-walk
            data2=pd.read_csv(qwalk_output)
            data2['glob_group_hybrid']=data2.apply(lambda x: x.group_q_prob if x.isCanonical=="NonCanonical" else x.global_q, axis=1)
            print(f"{dataset}|{database}|{file} 'glob_group_hybrid' range and NA: {min(data2['glob_group_hybrid'])} - {max(data2['glob_group_hybrid'])} and {col_isna(data2,'glob_group_hybrid')}")
            #INSERT HERE THRESHOLD BOOL?
            data2.to_csv(qwalk_output)
            
#            break
#    break

PXD014258|openprot|ESC-HF-Sample-BT474_openprot_PeptideProphet.pep.xml N of discoveries: 48292
PXD014258|openprot|ESC-HF-Sample-BT474_openprot_PeptideProphet.pep.xml targets and decoys and NA: 94.1771 and 5.8229 and 0.0
PXD014258|openprot|ESC-HF-Sample-BT474_openprot_PeptideProphet.pep.xml peptideprophet_probability range and NA: 0.05 - 1.0 and 0.0
PXD014258|openprot|ESC-HF-Sample-BT474_openprot_PeptideProphet.pep.xml 'global_q' range and NA: 0.0 - 0.06182937554969217 and 0.0
PXD014258|openprot|ESC-HF-Sample-BT474_openprot_PeptideProphet.pep.xml 'custom_q' range and NA: 0.0 - 0.8256410256410256 and 0.0
PXD014258|openprot|ESC-HF-Sample-BT474_openprot_PeptideProphet.pep.xml 'glob_cust_hybrid' range and NA: 0.0 - 0.8256410256410256 and 0.0
PXD014258|openprot|ESC-HF-Sample-BT474_openprot_PeptideProphet.pep.xml Canonical_Unmodified 70.1193%, Canonical_Expected 14.286%, Contam_Unmodified 6.6367%, NonCanonical_Unmodified 3.6859%, NonCanonical_Expected 3.3732%, Contam_Expected 1.8989%, NA 0.0


/home/vvshazia/miniconda3/envs/general/lib/python3.13/site-packages/pyteomics/auxiliary/target_decoy.py:83: RuntimeWarning: divide by zero encountered in divide
  q = tfalse / (ind - cumsum) / ratio


PXD005833|openprot|AM21_openprot_PeptideProphet.pep.xml 'custom_q' range and NA: 0.0 - 0.9607843137254902 and 0.0
PXD005833|openprot|AM21_openprot_PeptideProphet.pep.xml 'glob_cust_hybrid' range and NA: 0.00140964195094446 - 0.9607843137254902 and 0.0
PXD005833|openprot|AM21_openprot_PeptideProphet.pep.xml Canonical_Unmodified 57.4723%, Canonical_Expected 24.4801%, NonCanonical_Expected 8.1888%, NonCanonical_Unmodified 5.3874%, Contam_Unmodified 3.3725%, Contam_Expected 1.099%, NA 0.0
Run Group-walk on /project/def-marie87/vvshazia/pride_reanalysis/CompOmics/oui-discovery/oui-discovery-vv-data/processed/tpp_pride_reanalysis/tpp_pride_reanalysis_arch230525/PXD005833/PXD005833-openprot/AM21_openprot_PeptideProphet.csv
PXD005833|openprot|AM21_openprot_PeptideProphet.pep.xml 'glob_group_hybrid' range and NA: 0.0014096419509444 - 0.115638898906119 and 0.0
PXD005833|openprot|AM20_openprot_PeptideProphet.pep.xml N of discoveries: 10422
PXD005833|openprot|AM20_openprot_PeptideProphet.pep.xml t

/home/vvshazia/miniconda3/envs/general/lib/python3.13/site-packages/pyteomics/auxiliary/target_decoy.py:83: RuntimeWarning: divide by zero encountered in divide
  q = tfalse / (ind - cumsum) / ratio


PXD005833|openprot|AM11_openprot_PeptideProphet.pep.xml 'custom_q' range and NA: 0.0 - 1.086848635235732 and 0.0
PXD005833|openprot|AM11_openprot_PeptideProphet.pep.xml 'glob_cust_hybrid' range and NA: 0.0015188912094171254 - 1.086848635235732 and 0.0
PXD005833|openprot|AM11_openprot_PeptideProphet.pep.xml Canonical_Unmodified 63.2614%, Canonical_Expected 21.1176%, NonCanonical_Expected 7.6916%, Contam_Unmodified 3.6126%, NonCanonical_Unmodified 3.1553%, Contam_Expected 1.1615%, NA 0.0
Run Group-walk on /project/def-marie87/vvshazia/pride_reanalysis/CompOmics/oui-discovery/oui-discovery-vv-data/processed/tpp_pride_reanalysis/tpp_pride_reanalysis_arch230525/PXD005833/PXD005833-openprot/AM11_openprot_PeptideProphet.csv
PXD005833|openprot|AM11_openprot_PeptideProphet.pep.xml 'glob_group_hybrid' range and NA: 0.0015188912094171 - 0.0865461049284579 and 0.0
PXD005833|openprot|AM13_openprot_PeptideProphet.pep.xml N of discoveries: 11051
PXD005833|openprot|AM13_openprot_PeptideProphet.pep.xml

...

PXD005833|openprot|AM17_openprot_PeptideProphet.pep.xml 'custom_q' range and NA: 0.0 - 0.990521327014218 and 0.0
PXD005833|openprot|AM17_openprot_PeptideProphet.pep.xml 'glob_cust_hybrid' range and NA: 0.0015677491601343784 - 0.990521327014218 and 0.0
PXD005833|openprot|AM17_openprot_PeptideProphet.pep.xml Canonical_Unmodified 60.4896%, Canonical_Expected 21.6605%, NonCanonical_Expected 8.9409%, Contam_Unmodified 4.0766%, NonCanonical_Unmodified 3.5764%, Contam_Expected 1.256%, NA 0.0
Run Group-walk on /project/def-marie87/vvshazia/pride_reanalysis/CompOmics/oui-discovery/oui-discovery-vv-data/processed/tpp_pride_reanalysis/tpp_pride_reanalysis_arch230525/PXD005833/PXD005833-openprot/AM17_openprot_PeptideProphet.csv
PXD005833|openprot|AM17_openprot_PeptideProphet.pep.xml 'glob_group_hybrid' range and NA: 0.0015677491601343 - 0.0974071478626489 and 0.0
PXD005833|openprot|AM15_openprot_PeptideProphet.pep.xml N of discoveries: 10219
PXD005833|openprot|AM15_openprot_PeptideProphet.pep.xml 

/home/vvshazia/miniconda3/envs/general/lib/python3.13/site-packages/pyteomics/auxiliary/target_decoy.py:83: RuntimeWarning: divide by zero encountered in divide
  q = tfalse / (ind - cumsum) / ratio


PXD005833|openprot|AM15_openprot_PeptideProphet.pep.xml 'custom_q' range and NA: 0.0 - 0.9455958549222798 and 0.0
PXD005833|openprot|AM15_openprot_PeptideProphet.pep.xml 'glob_cust_hybrid' range and NA: 0.0016347501167678655 - 0.9455958549222798 and 0.0
PXD005833|openprot|AM15_openprot_PeptideProphet.pep.xml Canonical_Unmodified 59.6732%, Canonical_Expected 24.8459%, NonCanonical_Expected 7.3491%, NonCanonical_Unmodified 4.0415%, Contam_Unmodified 2.7009%, Contam_Expected 1.3896%, NA 0.0
Run Group-walk on /project/def-marie87/vvshazia/pride_reanalysis/CompOmics/oui-discovery/oui-discovery-vv-data/processed/tpp_pride_reanalysis/tpp_pride_reanalysis_arch230525/PXD005833/PXD005833-openprot/AM15_openprot_PeptideProphet.csv
PXD005833|openprot|AM15_openprot_PeptideProphet.pep.xml 'glob_group_hybrid' range and NA: 0.0016347501167678 - 0.0999892368959208 and 0.0
PXD005833|openprot|AM14_openprot_PeptideProphet.pep.xml N of discoveries: 10775
PXD005833|openprot|AM14_openprot_PeptideProphet.pep.x

/home/vvshazia/miniconda3/envs/general/lib/python3.13/site-packages/pyteomics/auxiliary/target_decoy.py:83: RuntimeWarning: divide by zero encountered in divide
  q = tfalse / (ind - cumsum) / ratio


PXD005833|openprot|AM14_openprot_PeptideProphet.pep.xml 'custom_q' range and NA: 0.0 - 0.8837837837837837 and 0.0
PXD005833|openprot|AM14_openprot_PeptideProphet.pep.xml 'glob_cust_hybrid' range and NA: 0.002029573789504204 - 0.8837837837837837 and 0.0
PXD005833|openprot|AM14_openprot_PeptideProphet.pep.xml Canonical_Unmodified 60.065%, Canonical_Expected 26.71%, NonCanonical_Expected 6.4687%, NonCanonical_Unmodified 3.406%, Contam_Unmodified 2.1995%, Contam_Expected 1.1508%, NA 0.0
Run Group-walk on /project/def-marie87/vvshazia/pride_reanalysis/CompOmics/oui-discovery/oui-discovery-vv-data/processed/tpp_pride_reanalysis/tpp_pride_reanalysis_arch230525/PXD005833/PXD005833-openprot/AM14_openprot_PeptideProphet.csv
PXD005833|openprot|AM14_openprot_PeptideProphet.pep.xml 'glob_group_hybrid' range and NA: 0.0020295737895042 - 0.0823623945359582 and 0.0
PXD005833|canon|AM10_canon_PeptideProphet.pep.xml N of discoveries: 10154
PXD005833|canon|AM10_canon_PeptideProphet.pep.xml targets and de

In [ ]:
#indicate pass of qval thresh per sample